#  LangChain의 RAG 콤포넌트 - 문서 임베딩(Embeddings) 

### **학습 목표:**  임베딩 모델과 벡터 데이터베이스를 효과적으로 연동할 수 있다

### **실습 자료**: 

- data/transformer.pdf

---

# 환경 설정 및 준비

`(1) Env 환경변수`

In [8]:
from dotenv import load_dotenv
load_dotenv()

True

`(2) 기본 라이브러리`

In [9]:
import os
from glob import glob  

from pprint import pprint  
import json

`(3) 문서 로드`

In [10]:
from langchain_community.document_loaders import PyPDFLoader

# PDF 로더 초기화
pdf_loader = PyPDFLoader('./data/transformer.pdf')

# 동기 로딩
pdf_docs = pdf_loader.load()
print(f'PDF 문서 개수: {len(pdf_docs)}')

C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_4080\3020463323.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


PDF 문서 개수: 15


`(4) 텍스트 분할`

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 텍스트 분할기 초기화
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,             # 청크 크기  
    chunk_overlap=200,           # 청크 중 중복되는 부분 크기
    length_function=len,         # 글자 수를 기준으로 분할
    separators=["\n\n", "\n", " ", ""],  # 구분자 - 재귀적으로 순차적으로 적용 
)

# PDF 문서를 텍스트로 분할
chunks = text_splitter.split_documents(pdf_docs)
print(f"생성된 텍스트 청크 수: {len(chunks)}")
print(f"각 청크의 길이: {list(len(chunk.page_content) for chunk in chunks)}")

생성된 텍스트 청크 수: 52
각 청크의 길이: [986, 910, 975, 452, 933, 995, 902, 907, 996, 385, 924, 954, 216, 924, 901, 950, 995, 913, 908, 870, 945, 973, 946, 997, 196, 980, 980, 946, 938, 999, 943, 920, 734, 958, 946, 945, 617, 983, 988, 994, 624, 944, 909, 941, 914, 986, 925, 927, 847, 812, 815, 818]


# 문서 임베딩(Document Embedding)

- 개념: 
    - 텍스트를 벡터(숫자 배열)로 변환하는 과정
    - 문서의 의미적 특성을 수치화하여 컴퓨터가 이해하고 처리할 수 있는 형태로 변환 

- 목적:
    - 텍스트 간 유사도 계산 가능
    - 벡터 데이터베이스 저장 및 검색
    - 의미 기반 문서 검색 구현

- LangChain의 임베딩 모델 종류:
    - OpenAI 임베딩
    - HuggingFace 임베딩 
    - Ollama 임베딩

### 1. **OpenAI**

- LangChain에서 가장 널리 사용되는 임베딩 모델 중 하나

- 주요 특징:
    1. 고품질의 임베딩 생성
    2. 다양한 언어 지원 (다국어 지원)
    3. 일관된 성능
    4. 손쉬운 통합

- 사용시 주의사항:
    1. API 키 설정이 필요 (환경 변수 OPENAI_API_KEY)
    2. API 사용량에 따른 비용 발생
    3. 긴 텍스트는 자동으로 분할되지 않으므로 필요시 TextSplitter를 사용


- 모델별 특징

    | 모델명 | 가격 효율 (페이지/1달러) | 성능 (MTEB) | 최대 입력 | 기본 차원 | 특징 및 추천 |
    | :--- | :---: | :---: | :---: | :---: | :--- |
    | **`text-embedding-3-small`** | **62,500** | 62.3% | 8,191 | 1,536 | **[가성비]** ada-002 대비 5배 저렴하고 성능 우수. 일반적인 RAG 구축 시 1순위. |
    | **`text-embedding-3-large`** | 9,615 | **64.6%** | 8,191 | 3,072 | **[고성능]** 미세한 의미 차이 구분이 중요할 때 사용. 차원 축소 기능 지원. |
    | **`text-embedding-ada-002`** | 12,500 | 61.0% | 8,191 | 1,536 | **[레거시]** 기존에 널리 쓰이던 모델이나, 현재는 v3-small 사용을 권장함. |

> **💡 Tip:** `text-embedding-3` 계열은 **Matryoshka Embedding** 기술이 적용되어, 저장 공간 절약을 위해 임베딩 차원(예: 1536 → 512)을 줄여서 요청해도 성능 하락이 매우 적습니다.

`(1) embedding 모델`

In [12]:
from langchain_openai import OpenAIEmbeddings

# OpenAIEmbeddings 모델 생성
embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-large",  # 사용할 모델 이름
    dimensions=None, # 원하는 임베딩 차원 수를 지정 가능 (기본값: None)
    )

# 임베딩 객체 출력
embeddings_model

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x000002D8071804D0>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x000002D807425D60>, model='text-embedding-3-large', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

In [13]:
# 임베딩 모델의 컨텍스트 길이 확인
embeddings_model.embedding_ctx_length

8191

In [14]:
# 임베딩 모델의 임베딩 차원 확인 - 기본값 (None)
embeddings_model.dimensions

In [15]:
# OpenAIEmbeddings 모델 생성할 때 임베딩 차원을 지정하는 예시
embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-small",  # 사용할 모델 이름
    dimensions=512, # 원하는 임베딩 차원 수를 지정 가능 (기본값: None)
    )

# 임베딩 모델의 임베딩 차원 확인 
embeddings_model.dimensions

512

In [16]:
# OpenAIEmbeddings 모델 생성
embeddings_openai = OpenAIEmbeddings(model="text-embedding-3-small")

# 임베딩 객체 출력
embeddings_openai

OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x000002D80C559D30>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x000002D80C55A420>, model='text-embedding-3-small', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

`(2) embed_documents 사용`

In [17]:
# 문서 컬렉션
documents = [
    "인공지능은 컴퓨터 과학의 한 분야입니다.",
    "머신러닝은 인공지능의 하위 분야입니다.",
    "딥러닝은 머신러닝의 한 종류입니다.",
    "자연어 처리는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.",
    "컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다."
]

# 문서 임베딩
document_embeddings_openai = embeddings_openai.embed_documents(documents)

# 임베딩 결과 출력
print(f"임베딩 벡터의 개수: {len(document_embeddings_openai)}")
print(f"임베딩 벡터의 차원: {len(document_embeddings_openai[0])}")
print(document_embeddings_openai[0])

임베딩 벡터의 개수: 5
임베딩 벡터의 차원: 1536
[-0.0022602081298828125, 0.01212310791015625, -0.0024471282958984375, 0.01502227783203125, 0.0184478759765625, -0.045684814453125, -0.0031032562255859375, 0.046966552734375, -0.01824951171875, -0.031829833984375, 0.006900787353515625, 0.00699615478515625, -0.018585205078125, -0.027099609375, 0.00498199462890625, -0.015594482421875, -0.041656494140625, 0.01003265380859375, 0.05157470703125, -0.045806884765625, -0.0146331787109375, -0.027801513671875, -0.018768310546875, -0.0227203369140625, 0.004375457763671875, -0.03973388671875, 0.05023193359375, 0.0177001953125, 0.0003368854522705078, -0.02337646484375, 0.0489501953125, -0.01526641845703125, -0.02996826171875, -0.061676025390625, 0.020538330078125, 0.03546142578125, 0.0025959014892578125, -0.006885528564453125, -0.005100250244140625, 0.0247344970703125, 0.007564544677734375, 0.0275115966796875, -0.0025386810302734375, 0.0260772705078125, -0.0179443359375, 0.005462646484375, -0.0258331298828125, 0.003633

`(3) embed_query 사용`

In [18]:
embedded_query_openai = embeddings_openai.embed_query("인공지능이란 무엇인가요?")

# 쿼리 임베딩 결과 출력
print(f"쿼리 임베딩 벡터의 차원: {len(embedded_query_openai)}")
print(embedded_query_openai)

쿼리 임베딩 벡터의 차원: 1536
[-0.0224456787109375, 0.0222015380859375, 0.0003859996795654297, 0.00574493408203125, 0.0122528076171875, -0.044769287109375, -0.026275634765625, 0.035797119140625, -0.0025959014892578125, 0.01476287841796875, -0.0019893646240234375, 0.0009360313415527344, -0.004917144775390625, -0.0736083984375, 0.00899505615234375, -0.0154266357421875, -0.05987548828125, -0.02239990234375, 0.022491455078125, -0.06915283203125, -0.02923583984375, 0.023223876953125, -0.036865234375, 0.0041046142578125, 0.011077880859375, -0.052734375, 0.01071929931640625, 9.28044319152832e-05, -0.0108489990234375, -0.03424072265625, 0.0253143310546875, -0.018829345703125, -0.0015420913696289062, -0.058807373046875, 0.0498046875, -0.0034999847412109375, -0.0028629302978515625, -0.00855255126953125, -0.00460052490234375, 0.0228729248046875, -0.01197052001953125, 0.0379638671875, 0.0033168792724609375, 0.035400390625, -0.051361083984375, 0.034393310546875, -0.0285491943359375, 0.004730224609375, -0.013

`(4) 유사도 기반 검색`

In [19]:
from langchain_community.utils.math import cosine_similarity
import numpy as np

# 쿼리와 가장 유사한 문서 찾기 함수
def find_most_similar(
        query: str, 
        doc_embeddings: np.ndarray,
        embeddings_model  # 기본값 제거, 명시적 전달 강제
        ) -> tuple[str, float]:
    """
    쿼리와 가장 유사한 문서를 찾는 함수
    
    Args:
        query: 검색 쿼리 문자열
        doc_embeddings: 문서 임베딩 배열
        embeddings_model: 임베딩 모델 객체
    
    Returns:
        tuple: (가장 유사한 문서, 유사도 점수)
    """
    # 쿼리 임베딩: OpenAI 임베딩 사용 
    query_embedding = embeddings_model.embed_query(query)

    # 코사인 유사도 계산
    similarities = cosine_similarity([query_embedding], doc_embeddings)[0]

    # 가장 유사한 문서 인덱스 찾기
    most_similar_idx = np.argmax(similarities)

    # 가장 유사한 문서와 유사도 반환: 문서, 유사도
    return documents[most_similar_idx], similarities[most_similar_idx]

# 예제 쿼리
queries = [
    "인공지능이란 무엇인가요?",
    "딥러닝과 머신러닝의 관계는 어떻게 되나요?",
    "컴퓨터가 이미지를 이해하는 방법은?"
]

# 각 쿼리에 대해 가장 유사한 문서 찾기
for query in queries:
    most_similar_doc, similarity = find_most_similar(
        query, 
        document_embeddings_openai, 
        embeddings_model=embeddings_openai
        )
    print(f"쿼리: {query}")
    print(f"가장 유사한 문서: {most_similar_doc}")
    print(f"유사도: {similarity:.4f}")
    print()
    

쿼리: 인공지능이란 무엇인가요?
가장 유사한 문서: 인공지능은 컴퓨터 과학의 한 분야입니다.
유사도: 0.7114

쿼리: 딥러닝과 머신러닝의 관계는 어떻게 되나요?
가장 유사한 문서: 딥러닝은 머신러닝의 한 종류입니다.
유사도: 0.6826

쿼리: 컴퓨터가 이미지를 이해하는 방법은?
가장 유사한 문서: 컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.
유사도: 0.7054



### 2. **Huggingface**

- LangChain에서 오픈소스 기반의 대표적인 임베딩 모델

- 주요 특징:
    1. 로컬 환경에서 실행 가능
    2. 다양한 사전학습 모델 지원
    3. 커스텀 모델 학습 및 적용 가능
    4. 무료 사용 가능 (API 비용 없음)

- 사용시 주의사항:
    1. 로컬 컴퓨팅 자원 필요 (CPU/GPU)
    2. 초기 모델 다운로드 시간 소요
    3. 메모리 사용량 고려 필요
    4. transformers 라이브러리 설치 필요

- 임베딩 벡터 특성:
    1. 모델별로 다양한 차원 제공 (128 ~ 1024)
    2. sentence-transformers 기반 구현
    3. BERT 계열 모델 구조 사용
    4. 코사인 유사도 기반 검색 최적화

- 대표적인 임베딩 모델:

    | 모델명 (Hugging Face ID) | 차원 | 언어 | 특징 및 추천 용도 |
    | :--- | :---: | :---: | :--- |
    | **`all-MiniLM-L6-v2`** | 384 | **영어** | **[영어 표준/경량]** 매우 빠르고 메모리 효율이 좋음. 영어 전용 검색/분류 작업의 입문용 모델. |
    | **`all-mpnet-base-v2`** | 768 | **영어** | **[영어 고성능]** MiniLM보다 느리지만, 문장의 뉘앙스를 가장 정확하게 포착함. 영어권 RAG의 표준. |
    | **`paraphrase-multilingual-MiniLM-L12-v2`** | 384 | 다국어 | **[다국어 경량]** 한국어를 포함한 50개국어 지원. 속도가 빨라 실시간 서비스에 적합. |
    | **`intfloat/multilingual-e5-large`** | 1024 | 다국어 | **[다국어 고성능]** 다국어 벤치마크 상위권 모델. (사용 시 `query:`, `passage:` 접두어 필요) |
    | **`BAAI/bge-m3`** | 1024 | 다국어 | **[한국어 최적]** 한국어 처리 성능이 매우 뛰어나며, 긴 문장(8192 토큰)도 처리 가능. |


`(1) embedding 모델`

- langchain_huggingface 설치 필요

In [20]:
from langchain_huggingface import HuggingFaceEmbeddings  

# Hugging Face의 임베딩 모델 생성
embeddings_gemma = HuggingFaceEmbeddings(
    #model_name="google/embeddinggemma-300m",          # 사용할 모델 이름 - 구글의 경량 임베딩 모델
    model_name = "BAAI/bge-m3"
    # model_kwargs={'device': 'cuda'}  # GPU 사용시
    # model_kwargs={'device': 'mps'}   # Mac M1 사용시
)

# 임베딩 객체 출력
embeddings_gemma

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

HuggingFaceEmbeddings(model_name='BAAI/bge-m3', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

`(2) embed_documents 사용`

In [21]:
# 문서 임베딩
document_embeddings_gemma = embeddings_gemma.embed_documents(documents)

# 임베딩 결과 출력
print(f"임베딩 벡터의 개수: {len(document_embeddings_gemma)}")
print(f"임베딩 벡터의 차원: {len(document_embeddings_gemma[0])}")
print(document_embeddings_gemma[0])

임베딩 벡터의 개수: 5
임베딩 벡터의 차원: 1024
[-0.03941448777914047, 0.008764867670834064, -0.012681609019637108, 0.002453196095302701, -0.008944743312895298, -0.007383682299405336, -0.005377364810556173, -0.009055862203240395, 0.03291524574160576, 0.00604549003764987, -0.027012895792722702, -0.027740878984332085, 0.0004441229102667421, 0.03013654425740242, 0.0172427985817194, 0.017090313136577606, 0.025524910539388657, -0.021856088191270828, -0.011341256089508533, -0.05702263116836548, -0.0003016895498149097, 0.01354314386844635, -0.007450080942362547, 0.018574437126517296, 0.002894673030823469, 0.008630610071122646, -0.0007444930379278958, -0.02890409529209137, 0.02072780579328537, -0.020500576123595238, 0.00806986540555954, -0.02675420418381691, 0.00396305974572897, -0.01630391925573349, -0.07406219840049744, -0.03365032747387886, -0.02387150004506111, -0.034550074487924576, -0.0347859263420105, 0.005483011715114117, -0.0500335767865181, -0.0028035573195666075, -0.02314690500497818, -0.07491144537

`(3) embed_query 사용`

In [22]:
embedded_query = embeddings_gemma.embed_query("인공지능이란 무엇인가요?")

# 쿼리 임베딩 결과 출력
print(f"쿼리 임베딩 벡터의 차원: {len(embedded_query)}")
print(embedded_query)

쿼리 임베딩 벡터의 차원: 1024
[-0.03703910857439041, -0.00483801169320941, 0.002937300829216838, -0.015514614060521126, -0.0009442013106308877, -0.041501618921756744, -0.006574462167918682, 0.011289630085229874, 0.021614041179418564, 0.004928698763251305, -0.020340587943792343, 0.01690521091222763, -0.012874161824584007, 0.005518929101526737, 0.014988314360380173, 0.024228785187005997, 0.007369145750999451, -0.028049886226654053, -0.014939015731215477, -0.05185190215706825, -0.006705053150653839, -0.0092515479773283, -0.016980808228254318, 0.006491472478955984, 0.0529317669570446, 0.04813733324408531, -0.008069523610174656, -0.023171763867139816, 0.01814303547143936, -0.011328112334012985, -0.0042403703555464745, -0.006354718003422022, -0.0022717334795743227, 0.014329425990581512, -0.03563681244850159, -0.00815580878406763, -0.011798224411904812, -0.04542406648397446, -0.040732864290475845, 0.0022139111533761024, -0.012132275849580765, 0.01789613626897335, -0.01914466731250286, -0.04192440584301

`(4) 유사도 기반 검색`

In [ ]:
# 예제 쿼리
queries = [
    "인공지능이란 무엇인가요?",
    "딥러닝과 머신러닝의 관계는 어떻게 되나요?",
    "컴퓨터가 이미지를 이해하는 방법은?"
]

# 각 쿼리에 대해 가장 유사한 문서 찾기
for query in queries:
    most_similar_doc, similarity = find_most_similar(query, document_embeddings_gemma, embeddings_model=embeddings_gemma) 
    print(f"쿼리: {query}")
    print(f"가장 유사한 문서: {most_similar_doc}")
    print(f"유사도: {similarity:.4f}")
    print()

쿼리: 인공지능이란 무엇인가요?
가장 유사한 문서: 인공지능은 컴퓨터 과학의 한 분야입니다.
유사도: 0.7269

쿼리: 딥러닝과 머신러닝의 관계는 어떻게 되나요?
가장 유사한 문서: 딥러닝은 머신러닝의 한 종류입니다.
유사도: 0.7057

쿼리: 컴퓨터가 이미지를 이해하는 방법은?
가장 유사한 문서: 컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.
유사도: 0.6843



### 3. **Ollama (로컬 실행 최적화)**

LangChain에서 로컬 LLM 및 임베딩 모델을 가장 손쉽게 실행할 수 있는 플랫폼입니다. 외부 API 전송 없이 로컬 자원(GPU/CPU)만 사용하므로 데이터 보안과 비용 절감에 최적화되어 있습니다.

**주요 특징:**
1. **완전한 로컬 실행:** 데이터가 외부로 유출되지 않아 기업 내부(On-premise) 구축에 적합.
2. **빠른 추론 속도:** C++ 기반의 런타임과 양자화(Quantization) 기술로 최적화됨.
3. **간편한 배포:** Docker 기반으로 모델 설치 및 실행이 매우 간단함 (`ollama pull 모델명`).

**사용 시 주의사항:**
1. **서버 실행 필수:** 백그라운드에서 `ollama serve`가 실행 중이어야 함.
2. **리소스 관리:** 고성능 모델(Large) 사용 시 충분한 RAM/VRAM 필요.
3. **API 설정:** LangChain 등에서 호출 시 엔드포인트(`localhost:11434`) 확인 필요.

**대표적인 임베딩 모델 비교:**

| 모델명 (Model Tag) | 차원 | 언어 | 특징 및 추천 용도 |
| :--- | :---: | :---: | :--- |
| **`nomic-embed-text`** | 768 | 영어 | **[Ollama 표준]** 긴 문맥(8192 토큰)을 지원하며, 오픈소스 중 밸런스가 가장 우수함. |
| **`mxbai-embed-large`** | 1024 | 영어 | **[SOTA 성능]** MTEB 리더보드 상위권 모델. 검색 정확도가 매우 높음. |
| **`snowflake-arctic-embed`** | 1024 | 영어 | **[검색 최적화]** Snowflake사가 RAG 및 대규모 검색 작업에 특화하여 설계함. |
| **`bge-m3`** | 1024 | **다국어** | **[한국어 추천]** Ollama에서 사용 가능한 가장 강력한 다국어/한국어 모델. |
| **`all-minilm`** | 384 | 영어 | **[초경량]** 속도가 매우 빠르고 CPU 환경에서도 부담 없이 실행 가능. |

**임베딩 벡터 특성:**
1. **고정 차원:** 모델별로 384~1024의 고정된 벡터 차원을 가짐.
2. **자동 양자화:** 원본 모델을 4비트(Q4_0) 등으로 압축하여 메모리 사용량을 대폭 줄임.

`(1) embedding 모델`

- langchain_ollama 설치 필요

In [25]:
from langchain_ollama import OllamaEmbeddings 

# OllamaEmbeddings 모델 생성
# embeddings_ollama = OllamaEmbeddings(
#     model="nomic-embed-text",          # 사용할 모델 이름
#     base_url="http://localhost:11434"  # Ollama 서버 주소
# )
embeddings_ollama = OllamaEmbeddings(model="bge-m3")

# 임베딩 객체 출력
embeddings_ollama

OllamaEmbeddings(model='bge-m3', dimensions=None, validate_model_on_init=False, base_url=None, client_kwargs={}, async_client_kwargs={}, sync_client_kwargs={}, mirostat=None, mirostat_eta=None, mirostat_tau=None, num_ctx=None, num_gpu=None, keep_alive=None, num_thread=None, repeat_last_n=None, repeat_penalty=None, temperature=None, stop=None, tfs_z=None, top_k=None, top_p=None)

`(2) embed_documents 사용`

In [26]:
# 문서 컬렉션
documents = [
    "인공지능은 컴퓨터 과학의 한 분야입니다.",
    "머신러닝은 인공지능의 하위 분야입니다.",
    "딥러닝은 머신러닝의 한 종류입니다.",
    "자연어 처리는 컴퓨터가 인간의 언어를 이해하고 생성하는 기술입니다.",
    "컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다."
]

# 문서 임베딩
document_embeddings_ollama = embeddings_ollama.embed_documents(documents)

# 임베딩 결과 출력
print(f"임베딩 벡터의 개수: {len(document_embeddings_ollama)}")
print(f"임베딩 벡터의 차원: {len(document_embeddings_ollama[0])}")
print(document_embeddings_ollama[0])

임베딩 벡터의 개수: 5
임베딩 벡터의 차원: 1024
[-0.039358072, 0.008702901, -0.012888061, 0.0023466076, -0.00901781, -0.0074256267, -0.0055463025, -0.009002523, 0.032844312, 0.005993907, -0.027288828, -0.027926745, 0.0003519508, 0.03018343, 0.01718855, 0.017260976, 0.025449436, -0.021874785, -0.01153651, -0.05692356, -0.0001967674, 0.013516217, -0.0074523496, 0.018612562, 0.002939299, 0.008597583, -0.0008192026, -0.02903734, 0.020703252, -0.020601517, 0.008144086, -0.026868172, 0.0036214152, -0.016139098, -0.073873, -0.0336498, -0.023811348, -0.03433851, -0.034622014, 0.0054867906, -0.05005907, -0.0029225799, -0.023076579, -0.07501195, -0.011301668, -0.028943168, -0.03417867, -0.025467193, -0.061402455, 0.0135706905, 0.024445036, -0.031425722, 0.050555617, 0.012957757, -0.06394739, 0.02730083, 0.0070393295, 0.010378978, -0.064018734, -0.016185155, -0.020708734, 0.03019408, -0.006299269, -0.013124084, -0.00065097207, 0.046930864, 0.00788988, 0.029638702, -0.034799233, -0.036378015, 0.02149646, 0.0177360

`(3) embed_query 사용`

In [27]:
embedded_query = embeddings_ollama.embed_query("인공지능이란 무엇인가요?")

# 쿼리 임베딩 결과 출력
print(f"쿼리 임베딩 벡터의 차원: {len(embedded_query)}")
print(embedded_query)

쿼리 임베딩 벡터의 차원: 1024
[-0.03678166, -0.004919923, 0.0029096627, -0.015549973, -0.00093368784, -0.04167643, -0.0066839247, 0.011325715, 0.02156796, 0.0049055773, -0.020472378, 0.016803848, -0.012875568, 0.0054638, 0.014963262, 0.024277883, 0.0074384157, -0.02795743, -0.015165, -0.05180563, -0.0065965015, -0.009214095, -0.017135957, 0.0065705935, 0.05292644, 0.04805525, -0.008188731, -0.023172189, 0.018120566, -0.011381682, -0.0043520182, -0.0063732113, -0.0024457138, 0.014372488, -0.035449363, -0.008185352, -0.0117982915, -0.045265276, -0.040650945, 0.002140653, -0.012299659, 0.017710827, -0.01911537, -0.041928478, 0.0010280136, -0.03903421, -0.02444973, -0.024537358, -0.02217448, -0.004504096, 0.031540778, -0.048727874, 0.018010242, 0.025275705, 0.0023308266, 0.048545647, -0.008125695, 0.028087381, -0.0737391, -0.020732114, -0.026399517, -0.007485619, -0.038752872, -0.017367119, 0.019281926, 0.06370354, 0.020380827, -0.0118443975, -0.01853148, -0.040764417, 0.0024853, 0.041489888, -0.057

`(4) 유사도 기반 검색`

In [28]:
# 예제 쿼리
queries = [
    "인공지능이란 무엇인가요?",
    "딥러닝과 머신러닝의 관계는 어떻게 되나요?",
    "컴퓨터가 이미지를 이해하는 방법은?"
]

# 각 쿼리에 대해 가장 유사한 문서 찾기
for query in queries:
    most_similar_doc, similarity = find_most_similar(query, document_embeddings_ollama, embeddings_model=embeddings_ollama) 
    print(f"쿼리: {query}")
    print(f"가장 유사한 문서: {most_similar_doc}")
    print(f"유사도: {similarity:.4f}")
    print()

쿼리: 인공지능이란 무엇인가요?
가장 유사한 문서: 인공지능은 컴퓨터 과학의 한 분야입니다.
유사도: 0.7271

쿼리: 딥러닝과 머신러닝의 관계는 어떻게 되나요?
가장 유사한 문서: 딥러닝은 머신러닝의 한 종류입니다.
유사도: 0.7051

쿼리: 컴퓨터가 이미지를 이해하는 방법은?
가장 유사한 문서: 컴퓨터 비전은 컴퓨터가 디지털 이미지나 비디오를 이해하는 방법을 연구합니다.
유사도: 0.6837



# [실습 프로젝트]

1. OpenAI text-embedding-3-small 임베딩 모델을 초기화합니다. 
2. 임베딩 차원을 각각 512와 1536으로 구분하여 2개의 모델을 생성합니다. 
3. 아래 주어진 문장들의 임베딩을 생성합니다. (임베딩 차원이 512, 1536인 경우 각각 2개씩 생성)
   - 문장1: "인공지능은 현대 사회를 변화시키고 있다"
   - 문장2: "AI 기술이 우리의 미래를 바꾸고 있다"
4. 생성된 임베딩의 차원을 출력합니다. (임베딩 차원이 512, 1536인 경우를 각각 출력)
5. 두 문장 간의 코사인 유사도를 계산합니다. (임베딩 차원이 512, 1536인 경우를 각각 비교)
6. [추가] 임베딩 차원(512 vs 1536)에 따른 유사도 차이를 분석하고, 어떤 차원이 더 효과적인지 생각해보세요.

In [36]:
# 여기에 코드를 작성하세요.

from langchain_openai import OpenAIEmbeddings
from langchain_community.utils.math import cosine_similarity
import numpy as np

embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-large",
    dimensions=512,
)

embeddings_model_large = OpenAIEmbeddings(
    model="text-embedding-3-large",
    dimensions=1536,
)

document = [
    "인공지능은 현대 사회를 변화시키고 있다.",
    "AI 기술이 우리의 미래를 바꾸고 있다."
]

document_embeddings_openai = embeddings_model.embed_documents(document)
document_embeddings_openai_large = embeddings_model_large.embed_documents(document)

print(f"차원_512: {len(document_embeddings_openai[0])}")
print(f"차원_1536: {len(document_embeddings_openai_large[0])}")


embedded_qry = embeddings_model.embed_query("인공지능이란 무엇인가요?")
embedded_qry_large = embeddings_model_large.embed_query("인공지능이란 무엇인가요?")

qry = [
    "AI란 무엇인가요?",
    "딥러닝과 머신러닝의 관계는 어떻게 되나요?",
    "컴퓨터가 영상을 이해하는 방법은?"
]

for query in qry:
    most_similar_doc, similarity = find_most_similar(query, document_embeddings_openai, embeddings_model=embeddings_model)
    print(f"쿼리: {query}")
    print(f"유사문서: {most_similar_doc}")
    print(f"유사도: {similarity}")
    print()

for query in qry:
    most_similar_doc, similarity = find_most_similar(query, document_embeddings_openai_large, embeddings_model=embeddings_model_large)
    print(f"쿼리: {query}")
    print(f"유사문서: {most_similar_doc}")
    print(f"유사도: {similarity:.4f}")
    print()

차원_512: 512
차원_1536: 1536
쿼리: AI란 무엇인가요?
유사문서: 머신러닝은 인공지능의 하위 분야입니다.
유사도: 0.42494259499007125

쿼리: 딥러닝과 머신러닝의 관계는 어떻게 되나요?
유사문서: 인공지능은 컴퓨터 과학의 한 분야입니다.
유사도: 0.3516882857902023

쿼리: 컴퓨터가 영상을 이해하는 방법은?
유사문서: 머신러닝은 인공지능의 하위 분야입니다.
유사도: 0.2617771709571086

쿼리: AI란 무엇인가요?
유사문서: 머신러닝은 인공지능의 하위 분야입니다.
유사도: 0.4283

쿼리: 딥러닝과 머신러닝의 관계는 어떻게 되나요?
유사문서: 머신러닝은 인공지능의 하위 분야입니다.
유사도: 0.3350

쿼리: 컴퓨터가 영상을 이해하는 방법은?
유사문서: 머신러닝은 인공지능의 하위 분야입니다.
유사도: 0.2703



In [39]:
# 1. OpenAI text-embedding-3-small 임베딩 모델 초기화
# 2. 임베딩 차원을 각각 512와 1536으로 구분하여 2개의 모델 생성

# 임베딩 차원 512인 모델
embeddings_512 = OpenAIEmbeddings(
    model="text-embedding-3-small",
    dimensions=512
)

# 임베딩 차원 1536인 모델 (기본 차원)
embeddings_1536 = OpenAIEmbeddings(
    model="text-embedding-3-small",
    dimensions=1536
)

print("모델 초기화 완료")
print(f"embeddings_512 차원: {embeddings_512.dimensions}")
print(f"embeddings_1536 차원: {embeddings_1536.dimensions}")
# 3. 주어진 문장들의 임베딩 생성

# 문장 정의
sentence1 = "인공지능은 현대 사회를 변화시키고 있다"
sentence2 = "AI 기술이 우리의 미래를 바꾸고 있다"

# 임베딩 차원 512인 경우
embedding1_512 = embeddings_512.embed_query(sentence1)
embedding2_512 = embeddings_512.embed_query(sentence2)

# 임베딩 차원 1536인 경우
embedding1_1536 = embeddings_1536.embed_query(sentence1)
embedding2_1536 = embeddings_1536.embed_query(sentence2)

print("임베딩 생성 완료")

# 4. 생성된 임베딩의 차원 출력

print("=== 임베딩 차원 확인 ===")
print(f"\n[임베딩 차원 512인 경우]")
print(f"문장1 임베딩 차원: {len(embedding1_512)}")
print(f"문장2 임베딩 차원: {len(embedding2_512)}")

print(f"\n[임베딩 차원 1536인 경우]")
print(f"문장1 임베딩 차원: {len(embedding1_1536)}")
print(f"문장2 임베딩 차원: {len(embedding2_1536)}")

# 5. 두 문장 간의 코사인 유사도 계산

from langchain_community.utils.math import cosine_similarity

# 임베딩 차원 512인 경우의 유사도
similarity_512 = cosine_similarity([embedding1_512], [embedding2_512])[0][0]

# 임베딩 차원 1536인 경우의 유사도
similarity_1536 = cosine_similarity([embedding1_1536], [embedding2_1536])[0][0]

print("=== 코사인 유사도 비교 ===")
print(f"\n문장1: {sentence1}")
print(f"문장2: {sentence2}")
print(f"\n[임베딩 차원 512] 코사인 유사도: {similarity_512:.4f}")
print(f"[임베딩 차원 1536] 코사인 유사도: {similarity_1536:.4f}")
print(f"\n유사도 차이: {abs(similarity_512 - similarity_1536):.4f}")

모델 초기화 완료
embeddings_512 차원: 512
embeddings_1536 차원: 1536
임베딩 생성 완료
=== 임베딩 차원 확인 ===

[임베딩 차원 512인 경우]
문장1 임베딩 차원: 512
문장2 임베딩 차원: 512

[임베딩 차원 1536인 경우]
문장1 임베딩 차원: 1536
문장2 임베딩 차원: 1536
=== 코사인 유사도 비교 ===

문장1: 인공지능은 현대 사회를 변화시키고 있다
문장2: AI 기술이 우리의 미래를 바꾸고 있다

[임베딩 차원 512] 코사인 유사도: 0.5538
[임베딩 차원 1536] 코사인 유사도: 0.4955

유사도 차이: 0.0583
